In [2]:
import os

# Define the path to the main directory containing the dataset
dataset_dir = "C:/Users/duaas/QC_miniproject"

# Define the paths to the subdirectories for intact and damaged images
intact_dir = os.path.join(dataset_dir, "intact")
damaged_dir = os.path.join(dataset_dir, "damaged")


In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define data augmentation parameters for training data
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Rescale pixel values to [0, 1]
    rotation_range=20,  # Randomly rotate images by up to 20 degrees
    width_shift_range=0.2,  # Randomly shift images horizontally
    height_shift_range=0.2,  # Randomly shift images vertically
    shear_range=0.2,  # Randomly apply shear transformations
    zoom_range=0.2,  # Randomly zoom in and out of images
    horizontal_flip=True,  # Randomly flip images horizontally
    fill_mode='nearest'  # Fill in newly created pixels after rotation or shifting
)

# Define data augmentation parameters for validation data (only rescaling)
val_datagen = ImageDataGenerator(rescale=1./255)


C:\Users\duaas\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\__init__.py:177: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [4]:
# Generate batches of training data
train_generator = train_datagen.flow_from_directory(
    "C:/Users/duaas/Downloads/QC/intact",  # Path to the directory containing intact images
    target_size=(224, 224),  # Resize images to 224x224
    batch_size=32,
    class_mode='binary'  # Since it's a binary classification problem
)

# Generate batches of validation data
val_generator = val_datagen.flow_from_directory(
    "C:/Users/duaas/Downloads/QC/damaged",  # Path to the directory containing damaged images
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)


Found 200 images belonging to 2 classes.
Found 200 images belonging to 2 classes.


In [5]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

In [6]:
# Cell 2: Define dataset paths
root_dir = 'C:\\Users\\duaas\\Downloads\\QC'
intact_side_dir = os.path.join(root_dir, 'intact', 'side')
intact_top_dir = os.path.join(root_dir, 'intact', 'top')
damaged_side_dir = os.path.join(root_dir, 'damaged', 'side')
damaged_top_dir = os.path.join(root_dir, 'damaged', 'top')


In [7]:
def load_images(directory):
    images = []
    labels = []
    for filename in os.listdir(directory):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            img_path = os.path.join(directory, filename)
            img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            images.append(img_array)
            labels.append(0 if 'intact' in directory else 1)
    return images, labels

In [8]:
intact_side_images, intact_side_labels = load_images(intact_side_dir)
intact_top_images, intact_top_labels = load_images(intact_top_dir)
damaged_side_images, damaged_side_labels = load_images(damaged_side_dir)
damaged_top_images, damaged_top_labels = load_images(damaged_top_dir)

images = np.concatenate((intact_side_images, intact_top_images, damaged_side_images, damaged_top_images))
labels = np.concatenate((intact_side_labels, intact_top_labels, damaged_side_labels, damaged_top_labels))


In [9]:
base_model = ResNet50(weights='imagenet', include_top=False)

In [10]:
for layer in base_model.layers:
    layer.trainable = False

In [11]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [12]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [25]:
model.fit(images, labels, epochs=50, batch_size=32, validation_split=0.2)

Epoch 1/50
10/10 [==============================] - 11s 1s/step - loss: 0.0063 - accuracy: 1.0000 - val_loss: 4.8335 - val_accuracy: 0.2125
Epoch 2/50
10/10 [==============================] - 11s 1s/step - loss: 0.0072 - accuracy: 1.0000 - val_loss: 5.7483 - val_accuracy: 0.1250
Epoch 3/50
10/10 [==============================] - 14s 1s/step - loss: 0.0069 - accuracy: 1.0000 - val_loss: 5.0628 - val_accuracy: 0.2125
Epoch 4/50
10/10 [==============================] - 15s 2s/step - loss: 0.0064 - accuracy: 1.0000 - val_loss: 4.9317 - val_accuracy: 0.2125
Epoch 5/50
10/10 [==============================] - 11s 1s/step - loss: 0.0057 - accuracy: 1.0000 - val_loss: 5.0642 - val_accuracy: 0.2125
Epoch 6/50
10/10 [==============================] - 11s 1s/step - loss: 0.0054 - accuracy: 1.0000 - val_loss: 5.4695 - val_accuracy: 0.1625
Epoch 7/50
10/10 [==============================] - 11s 1s/step - loss: 0.0053 - accuracy: 1.0000 - val_loss: 4.9633 - val_accuracy: 0.2125
Epoch 8/50
10/10 [==

In [26]:
model.save('medicine_package_classification_model.h5')

In [27]:
test_loss, test_accuracy = model.evaluate(images, labels)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

13/13 [==============================] - 14s 1s/step - loss: 1.1983 - accuracy: 0.8375
Test Loss: 1.1982738971710205
Test Accuracy: 0.8374999761581421


In [3]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Load the pre-trained ResNet50 model
model = load_model('medicine_package_classification_model.h5')

# Define function to preprocess image
def preprocess_image(image_path):
    img = Image.open(image_path)
    img = img.resize((224, 224))  # Resize image to match model input shape
    img_array = np.array(img) / 255.0  # Normalize pixel values
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    return img_array

# Define function to classify image
def classify_image(image_path):
    img_array = preprocess_image(image_path)
    prediction = model.predict(img_array)
    if prediction[0] < 0.5:
        return "Intact"
    else:
        return "Damaged"

# Define function to open file dialog and classify selected image
def open_file_dialog():
    file_path = filedialog.askopenfilename(title="Select an image file", filetypes=(("Image files", "*.jpg;*.jpeg;*.png"), ("All files", "*.*")))
    if file_path:
        try:
            # Display selected image
            img = Image.open(file_path)
            img.thumbnail((300, 300))  # Resize image for display
            img = ImageTk.PhotoImage(img)
            img_label.configure(image=img)
            img_label.image = img
            
            # Classify image
            result = classify_image(file_path)
            result_label.config(text="Predicted Class: " + result)
        except Exception as e:
            messagebox.showerror("Error", "An error occurred: " + str(e))

# Create main window
window = tk.Tk()
window.title("Medicine Package Classification")
window.geometry("400x400")

# Create image label to display selected image
img_label = tk.Label(window)
img_label.pack(pady=10)

# Create button to open file dialog
open_button = tk.Button(window, text="Open Image", command=open_file_dialog)
open_button.pack(pady=10)

# Create label to display classification result
result_label = tk.Label(window, text="")
result_label.pack(pady=10)

# Run the application
window.mainloop()


In [35]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model

# Load pre-trained ResNet50 model
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

# Add a new top layer to the model
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
predictions = Dense(2, activation='softmax')(x)

# Create a new model with the new top layer
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Save the model
model.save('medicine_package_classification_model.h5')

In [2]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Load the pre-trained ResNet50 model
model = load_model('medicine_package_classification_model.h5')

# Define function to preprocess image
def preprocess_image(image_path):
    img = Image.open(image_path)
    img = img.resize((224, 224))  # Resize image to match model input shape
    img_array = np.array(img) / 255.0  # Normalize pixel values
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    return img_array

# Define function to classify image
def classify_image(image_path):
    img_array = preprocess_image(image_path)
    prediction = model.predict(img_array)
    if prediction[0][1] < 0.5:
        return "Intact"
    else:
        return "Damaged"

# Define function to open file dialog and classify selected image
def open_file_dialog():
    file_path = filedialog.askopenfilename(title="Select an image file", filetypes=(("Image files", "*.jpg;*.jpeg;*.png"), ("All files", "*.*")))
    if file_path:
        try:
            # Display selected image
            img = Image.open(file_path)
            img.thumbnail((300, 300))  # Resize image for display
            img = ImageTk.PhotoImage(img)
            img_label.configure(image=img)
            img_label.image = img
            
            # Classify image
            result = classify_image(file_path)
            result_label.config(text="Predicted Class: " + result)
        except Exception as e:
            messagebox.showerror("Error", "An error occurred: " + str(e))

# Create main window
window = tk.Tk()
window.title("Medicine Package Classification")
window.geometry("400x400")

# Create image label to display selected image
img_label = tk.Label(window)
img_label.pack(pady=10)

# Create button to open file dialog
open_button = tk.Button(window, text="Open Image", command=open_file_dialog)
open_button.pack(pady=10)

# Create label to display classification result
result_label = tk.Label(window, text="")
result_label.pack(pady=10)

# Run the application
window.mainloop()